# DAX Performance Investigation: BigQueryAdventureWorksDW
## Diagnosing Fact-Table Iteration & Context Transition Bottlenecks

This interactive notebook presents the empirical evidence, VPAX structural characteristics, DAX Studio Server Timings analysis, and the optimized pattern for **`Total Sales Red`** vs **`Total Sales Red BAD`**.

### 1. Executive Metrics Comparison

| Metric | Fast Measure (`Total Sales Red`) | Slow Measure (`Total Sales Red BAD`) | Variance |
| :--- | ---:| ---:| ---:|
| **Total Duration** | **4 ms** | **107 ms** | **26.8x slower** |
| **Storage Engine (SE) Duration** | **1 ms** | **73 ms** | **73x slower** |
| **Formula Engine (FE) Duration** | **3 ms** | **34 ms** | **11.3x slower** |
| **Storage Engine Queries** | **1** | **3** | **+2 queries** |
| **Materialized Rows** | **1 row** | **120,797 rows** | **120,797x more** |
| **Intermediate Datacache** | **1 KB** | **11,563 KB (~11.3 MB)** | **11,563x more** |

### 2. The Original DAX Definitions

```dax
// Fast Measure
measure 'Total Sales Red' = 
CALCULATE (
    [Total Sales],
    DimProduct[Color] = "Red"
)

// Slow Measure
measure 'Total Sales Red BAD' = 
SUMX (
    FactInternetSales,
    IF (
        RELATED ( DimProduct[Color] ) = "Red",
        CALCULATE ( [Total Sales] )
    )
)
```

### 3. VPAX Structural Evidence

- **`FactInternetSales`**: 60,398 rows, Total Size: 1.99 MB
- **`DimProduct`**: 606 rows, Total Size: 1.33 MB
- **`DimProduct[Color]`**: Cardinality = 10 distinct values
- **Relationship**: `FactInternetSales[ProductKey]` (Many) $\rightarrow$ `DimProduct[ProductKey]` (One), Active = True

> **Key Insight**: The slow measure's Storage Engine scans returned exactly 60,398 rows twice, matching the entire row count of `FactInternetSales`.

### 4. Root Cause Analysis

1. **Context Transition over Fact Table**: Calling `CALCULATE([Total Sales])` inside the row context of `FactInternetSales` converts all 22 columns of each row into a filter context 60,398 times.
2. **Excessive Materialization**: VertiPaq cannot aggregate to a scalar and must dump 120,797 uncompressed rows (11.5 MB) across process boundaries.
3. **Formula Engine Bottleneck**: Single-threaded FE processes 60,398 iterations of `IF` and `RELATED`.

### 5. Recommended Fix

```dax
measure 'Total Sales Red' = 
CALCULATE (
    [Total Sales],
    KEEPFILTERS ( DimProduct[Color] = "Red" )
)
```